# PathFinderShip — Focused Model Evidence v1

Bu kısa akış yalnızca proje tarafından fine-tune edilmiş **tek MiniLM** modeli ile `Primee/Models` altındaki **altı Flan-T5 Large LoRA denemesini** karşılaştırır. YOLO/COCO, IFEval, RAGBench, eski Small/Base ve beam-4 testleri çalıştırılmaz. Hücreleri yukarıdan aşağıya sırayla çalıştırın.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys

def find_bundle(start: Path) -> Path:
    for candidate in [start, *start.parents, *[p.parent for p in start.glob('**/UPLOAD_MANIFEST.json')]]:
        if (candidate / 'UPLOAD_MANIFEST.json').exists() and (candidate / 'benchmarks').exists():
            return candidate.resolve()
    raise FileNotFoundError('UPLOAD_MANIFEST.json bulunan bundle klasörü bulunamadı.')

BUNDLE_ROOT = find_bundle(Path.cwd())
studio_root = Path('/teamspace/studios/this_studio')
OUTPUT_BASE = (studio_root / 'pathfinder_outputs') if studio_root.exists() else (BUNDLE_ROOT / 'outputs')
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
active_file = OUTPUT_BASE / 'ACTIVE_FOCUSED_RUN_ID.txt'
RUN_ID = active_file.read_text(encoding='utf-8').strip() if active_file.exists() else datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
active_file.write_text(RUN_ID, encoding='utf-8')
RUN_DIR = OUTPUT_BASE / f'pathfinder_focused_results_{RUN_ID}'
DATA_DIR = BUNDLE_ROOT / 'benchmarks' / 'data'
MODELS_ROOT = BUNDLE_ROOT / 'models'
MANIFEST = BUNDLE_ROOT / 'benchmarks' / 'config' / 'focused_evidence.yaml'
os.environ['PYTHONPATH'] = str(BUNDLE_ROOT) + os.pathsep + os.environ.get('PYTHONPATH', '')
print('BUNDLE_ROOT =', BUNDLE_ROOT)
print('RUN_DIR     =', RUN_DIR)

## 1. Bağımlılıkları kur
Lightning'ın CUDA uyumlu Torch kurulumu korunur.

In [ ]:
requirements = BUNDLE_ROOT / 'benchmarks' / 'lightning' / 'requirements-lightning.txt'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])
print('Bağımlılıklar kuruldu.')

## 2. GPU ve yükleme paketini doğrula
`errors = []` görülmeden teste geçmeyin.

In [ ]:
import hashlib
subprocess.run(['nvidia-smi'], check=False)
upload_manifest = json.loads((BUNDLE_ROOT / 'UPLOAD_MANIFEST.json').read_text(encoding='utf-8'))
errors = []
for item in upload_manifest['files']:
    path = BUNDLE_ROOT / item['relative_path']
    if not path.exists():
        errors.append((item['relative_path'], 'missing'))
    elif hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
        errors.append((item['relative_path'], 'sha256_mismatch'))
assert not errors, f'Yükleme paketi doğrulanamadı: {errors[:10]}'
print(f"Upload doğrulandı: {upload_manifest['file_count']} dosya; errors = {errors}")

## 3. Test yardımcısı
Her model ayrı durum dosyası yazar; bir deney hata alırsa diğerleri devam eder. H100 için batch 16 kullanılır.

In [ ]:
def run_experiment(experiment_id: str, force: bool = False):
    command = [
        sys.executable, '-m', 'benchmarks.run',
        '--manifest', str(MANIFEST), '--models-root', str(MODELS_ROOT),
        '--data-dir', str(DATA_DIR), '--run-dir', str(RUN_DIR),
        '--experiment', experiment_id, '--generation-batch-size', '16',
        '--skip-beam-comparison', '--onnx-parity-examples-per-suite', '25',
    ]
    if force:
        command.append('--force')
    print('RUN:', experiment_id)
    subprocess.run(command, cwd=BUNDLE_ROOT, check=False)
    status_path = RUN_DIR / 'status' / f'{experiment_id}.json'
    print(status_path.read_text(encoding='utf-8') if status_path.exists() else 'Durum dosyası oluşmadı.')

## 4. MiniLM — tek eğitilmiş sürüm
1.000 dengeli intent örneği; accuracy, macro/weighted F1, sınıf metrikleri, calibration, CPU latency ve confusion matrix.

In [ ]:
run_experiment('minilm_intent_int8')

## 5. Flan-T5 Large LoRA deney zinciri
Altı adapter aynı dondurulmuş 300 Chat + 160 proje RAG örneğinde greedy decoding ile sınanır. `kötü`, ara checkpoint'ler, First Try ve final aday birlikte tutulur.

In [ ]:
for experiment_id in [
    'flan_large_lora_qv_failed',
    'flan_large_lora_rag2_step1320',
    'flan_large_lora_chat12_step1485',
    'flan_large_lora_chat12_step1980',
    'flan_large_lora_first_try',
    'flan_large_lora_second_try',
]:
    run_experiment(experiment_id)

## 6. Raporu ve görselleri üret
Tabloda başarısız/ara/final bütün modeller yer alır. En iyi Chat ve RAG sonucu görev bazında yeşil ve `BEST` etiketiyle gösterilir; yanıltıcı birleşik skor üretilmez.

In [ ]:
ZIP_PATH = OUTPUT_BASE / f'{RUN_DIR.name}.zip'
subprocess.check_call([
    sys.executable, '-m', 'benchmarks.report', '--run-dir', str(RUN_DIR),
    '--zip-path', str(ZIP_PATH),
    '--historical-evidence', str(BUNDLE_ROOT / 'historical_evidence' / 'historical_extraction.json'),
    '--experiment-manifest', str(MANIFEST),
], cwd=BUNDLE_ROOT)
print('TAMAMLANDI')
print('İndirilecek ZIP:', ZIP_PATH)
print('ZIP dosyasını Windows üzerindeki lightning_results_incoming klasörüne koyun.')